In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path so 'src' is importable
repo_root = str(Path.cwd().parents[1]) if Path.cwd().name == 'research_questions' else str(Path.cwd())
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

In [ ]:
from src.analysis.get_results import get_pandas_dataset
from src.analysis.dataset_graphs import plot_graphs_of_dataset_loc
from src.analysis.get_tables_results import create_table_cleaned
from src.analysis.get_results import bar_chart_fix_position_cleaned
from src.analysis.get_results import sucess_vs_position_cleaned, get_latex_table_with_verif_stats
from src.analysis.get_results import compute_stats_tests

import src.config as gl
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [ ]:
RESULT_DIR = gl.BASE_PATH / "results/dafny_llm_results_rq3__example_gatherer"
DATASET_DIR = gl.DAFNY_ASSERTION_DATASET

print(DATASET_DIR)
print(RESULT_DIR)
verif_data_pd = get_pandas_dataset(DATASET_DIR, RESULT_DIR)
verif_data_pd  = verif_data_pd.assign(success=lambda d: d['verif_sucess'] > 0) 

In [ ]:
# Graphs of position evaluation
test_models = {
    "claude-haiku-4.5__loc_ORACLE_inf_LLM": "NoEx$_{in}$",
    "claude-haiku-4.5__loc_ORACLE_inf_LLM_EXAMPLE_sExInf_RANDOM_nExInf_3": "Random$_{in}$",
    "claude-haiku-4.5__loc_ORACLE_inf_LLM_EXAMPLE_sExInf_DYNAMIC_nExInf_3_aExInf_1.0": "MulEmb1.00$_{in}$",
    "claude-haiku-4.5__loc_ORACLE_inf_LLM_EXAMPLE_sExInf_EMBEDDED_nExInf_3": "Embed$_{in}$",
    "claude-haiku-4.5__loc_ORACLE_inf_LLM_EXAMPLE_sExInf_TFIDF_nExInf_3": "Tfidf$_{in}$",
    "claude-haiku-4.5__loc_ORACLE_inf_LLM_EXAMPLE_sExInf_DYNAMIC_nExInf_3_aExInf_0.5": "MulEmb0.50$_{in}$",
    "claude-haiku-4.5__loc_ORACLE_inf_LLM_EXAMPLE_sExInf_DYNAMIC_nExInf_3_aExInf_0.75": "MulEmb0.75$_{in}$",
    "claude-haiku-4.5__loc_ORACLE_inf_LLM_EXAMPLE_sExInf_DYNAMIC_nExInf_3_aExInf_0.25": "MulEmb0.25$_{in}$",
}

# Filter if interested in per benchmark test
#verif_data_pd = verif_data_pd[verif_data_pd["benchmark"] != "w/o-1"]

dual_name = "rq3__example_gatherer_sucess_vs_position.pdf"
images_p = gl.BASE_PATH / "images"

info = get_latex_table_with_verif_stats(verif_data_pd, "Verification success rate for each approach for each category of benchmarks for the example retrieval strategy on the ground truth positions (\lgroundT{}).", "tbl:assertion-inference-verification-example", test_models)
print(info)
sucess_vs_position_cleaned(verif_data_pd ,"DOUBLE",   test_models, images_p / dual_name)

In [ ]:
verif_renamed = verif_data_pd.copy()
verif_renamed["llm"] = verif_renamed["llm"].map(test_models)

print("##### Comparing NoEx$_{in}$ with Random$_{in} #####")
compute_stats_tests("NoEx$_{in}$","Random$_{in}$", verif_renamed)

print("##### Comparing MulEmb0.25$_{in}$ with NoEx$_{in} #####\n")
compute_stats_tests("NoEx$_{in}$","MulEmb0.25$_{in}$", verif_renamed)

print("##### Comparing MulEmb0.25$_{in}$ with Tfidf$_{in}$ #####\n")
compute_stats_tests("Tfidf$_{in}$","MulEmb0.25$_{in}$", verif_renamed)